# Module 3 — Morphometry, texture and temporal pupil response
Full-video configurable tracking with an explicitly configured initial pupil center, timestamp-based curves, artifact flags and exploratory response parameters. Review overlays and manual errors before interpreting physiological estimates. An accepted mask passes heuristic checks; it is not guaranteed anatomically correct.

In [ ]:
import sys, pathlib, json
ROOT=pathlib.Path.cwd().resolve()
if not (ROOT / "config.yaml").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import morphology
from psib.config import load_config
from psib import video_io as vio, roi as R
from psib.calibration import pixels_per_cm, select_points
from psib.segmentation import segment_methods, morphology_variants
from psib.features import descriptors
from psib.temporal import track_video, clean_curve, response_parameters
from psib.stimulus import detect_stimulus_onset
from psib.plotting import save_fig
cfg=load_config(ROOT / "config.yaml"); m3=cfg["module3"]; m2=cfg["module2"]
video=pathlib.Path(cfg["video_path"])
out=cfg["_output_dir"] / "module3" / video.stem
out.mkdir(parents=True,exist_ok=True)
cal=cfg["calibration"]; scale=None
if cal["enabled"]:
    image,cal_t,_=vio.read_frame_at_time(video,cal["frame_time_s"])
    points=select_points(image,title=f"Select ruler endpoints {cal['distance_cm']} cm apart") if cal["mode"]=="interactive" else cal["points_xy"]
    scale=pixels_per_cm(points,cal["distance_cm"])
    (out / "calibration.json").write_text(json.dumps({"video":str(video),"timestamp_s":cal_t,"points_xy":points,"distance_cm":cal["distance_cm"],"pixels_per_cm":scale},indent=2))
if scale is None:
    stale=out/'calibration.json'
    if stale.exists(): stale.unlink()
print("Scale (px/cm):",scale)


## 3.1 Frame descriptors
Shape: area, perimeter, centroid, circularity and eccentricity. RGB first-order statistics: mean, standard deviation, quantized entropy and skewness. Symmetric GLCM pools four directions and distances 1 and 2; only pixel pairs with both endpoints inside the region are counted. Constant regions have undefined correlation/skewness.

The iris texture sample is an approximate 20-pixel ring outside the pupil mask, not an anatomical iris segmentation. Inspect it for reflections, eyelids or sclera before discussing iris texture. Calibration converts lengths and areas, while texture is calculated in pixel space.

In [ ]:
times=[cfg["frames"]["primary_time_s"],cfg["frames"]["secondary_time_s"]]
frames=vio.read_frames_at_times(video,times)
roi=R.get_roi(cfg,frames[0][0]); features=[]
for index,((frame,t,idx),init) in enumerate(zip(frames,m2["initializations"])):
    rgb=R.crop(frame,roi)
    masks=segment_methods(rgb,**init,clusters=m2["clusters"],snake_iterations=m2["snake_iterations"])
    pupil=morphology_variants(masks[m2["method"]],m2["morphology_radius"],m2["min_component_size"])["opening_closing_filled"]
    ring=morphology.dilation(pupil,morphology.disk(20)) & ~morphology.dilation(pupil,morphology.disk(3))
    fig,ax=plt.subplots(); ax.imshow(rgb); ax.contour(pupil,levels=[.5],colors='red'); ax.contour(ring,levels=[.5],colors='yellow')
    ax.set_title(f"Pupil (red) / approximate iris sample (yellow), {t:.3f} s")
    save_fig(fig,f"m3_texture_regions_{index}",cfg); plt.show()
    for region,mask in [('pupil',pupil),('approximate_iris_ring',ring)]:
        features.append({'timestamp_s':t,'frame_index':idx,'region':region,**descriptors(rgb,mask,scale,m3['texture_levels'])})
features=pd.DataFrame(features); features.to_csv(out / 'frame_features.csv',index=False)
display(features)


## 3.2 Full-video curve
The tracker uses the first configured center as a spatial prior. Snakes reacquire the nearest dark K-means component to the configured pupil seed on each frame, then use a fresh circle of fixed radius at its centroid. This avoids propagating gradual drift. Alternatives are K-means (K=5 in Lab) and watershed; select a method after reviewing its overlays. It uses raw presentation timestamps. Masks touching the crop boundary, exceeding area bounds, jumping in diameter, moving too quickly or lacking pupil/background contrast are rejected. Rejected raw measurements remain in the CSV with reasons. These checks can miss gradual drift or a blink that still resembles a pupil.

Only short internal gaps are interpolated. Long gaps and endpoints remain NaN. A time-window median smooths each finite segment separately. Overlays are sampled through the full recording for review; tracking is still experimental on A_1 because of its reflection.

In [ ]:
raw,previews=track_video(video,roi,**m2['initializations'][0],options=m3,scale=scale)
curve=clean_curve(raw,m3['max_gap_s'],m3['smooth_window_s'])
if scale is not None:
    curve['diameter_clean_mm']=10*curve['diameter_clean_px']/scale
    curve['diameter_smooth_mm']=10*curve['diameter_smooth_px']/scale
    curve['area_smooth_mm2']=100*curve['area_smooth_px2']/scale**2
curve.to_csv(out / 'pupil_curve.csv',index=False)
print(f"Decoded {len(curve)} frames; accepted {curve.accepted.mean():.1%}; interpolated {curve.interpolated.sum()} frames")
fig,axes=plt.subplots(2,1,figsize=(12,7),sharex=True)
axes[0].plot(curve.timestamp_s,curve.equivalent_diameter_px,alpha=.35,label='Raw')
axes[0].plot(curve.timestamp_s,curve.diameter_smooth_px,label='Accepted/interpolated/smoothed')
axes[0].scatter(curve.loc[~curve.accepted,'timestamp_s'],curve.loc[~curve.accepted,'equivalent_diameter_px'],color='red',s=12,label='Rejected')
axes[0].set_ylabel('Equivalent diameter (px)'); axes[0].legend()
axes[1].plot(curve.timestamp_s,curve.area_smooth_px2); axes[1].set_ylabel('Area (px²)'); axes[1].set_xlabel('Time (s)')
fig.tight_layout(); save_fig(fig,'m3_temporal_curve',cfg); plt.show()
fig,axes=plt.subplots(int(np.ceil(len(previews)/4)),4,figsize=(16,3*int(np.ceil(len(previews)/4))),squeeze=False)
for ax in axes.ravel():ax.axis('off')
for ax,(idx,t,img,mask,valid) in zip(axes.ravel(),previews):
    ax.imshow(img); ax.contour(mask,levels=[.5],colors='lime' if valid else 'red')
    ax.set_title(f"#{idx} {t:.2f}s {'accepted' if valid else 'rejected'}")
fig.tight_layout();save_fig(fig,'m3_tracking_review',cfg);plt.show()


## 3.3 Response estimates
Confirm the stimulus visually and set `module3.stimulus_time_s`; otherwise a global-brightness estimate is used and labelled as such. A records use constriction; B records use redilation. Latency is the first sustained change exceeding both 5% of the baseline diameter and 3 baseline robust standard deviations (configurable). Peak speeds come from adjacent smoothed samples and remain sensitive to segmentation error.

Insufficient data produces NaN. Long gaps cannot count toward a sustained response. These are algorithmic estimates, not validated physiological measurements.

In [ ]:
kind=m3['record_type'] or vio.record_type(video)
source='manual' if m3['stimulus_time_s'] is not None else 'brightness_estimate'
if source=='manual': stimulus=float(m3['stimulus_time_s'])
else:
    scan=vio.scan_video(video)
    stimulus=detect_stimulus_onset(scan['timestamps_s'],scan['brightness'],kind,cfg['module0']['brightness_smooth_frames'])['onset_s']
params=response_parameters(curve,stimulus,kind,m3,scale)
params.update(video=video.name,stimulus_source=source,pixels_per_cm=scale)
pd.DataFrame([params]).to_csv(out / 'physiological_parameters.csv',index=False)
display(pd.DataFrame([params]))


## 3.4 Manual validation and variability
Optional CSV columns: `frame_index`, `area_px2` and/or `equivalent_diameter_px`. Compare automatic raw measurements on accepted frames with manual measurements of the same frames/crop. Rejected frames are counted separately. Metrics remain in pixels, avoiding mixed calibration units.

Run this notebook for each available recording with its own ROI, initialization and ruler calibration. The final table aggregates saved per-video outputs by A/B; sample variability needs at least two recordings. No extra recordings are fabricated.

In [ ]:
manual_path=m3['manual_measurements_csv']
if manual_path:
    p=pathlib.Path(manual_path); manual=pd.read_csv(p if p.is_absolute() else ROOT/p)
    if 'frame_index' not in manual or manual.frame_index.duplicated().any(): raise ValueError('Manual frame indices must be present and unique')
    if not manual.frame_index.isin(curve.frame_index).all(): raise ValueError('Manual frame indices must exist in the current video')
    merged=manual.merge(curve,on='frame_index',suffixes=('_manual','_auto'),validate='one_to_one')
    errors=[]
    for metric in ['area_px2','equivalent_diameter_px']:
        if metric not in manual:continue
        use=merged.accepted & np.isfinite(merged[metric+'_manual']) & np.isfinite(merged[metric+'_auto'])
        e=(merged.loc[use,metric+'_auto']-merged.loc[use,metric+'_manual']).to_numpy()
        errors.append({'metric':metric,'n_matched':len(merged),'n_evaluated':len(e),'n_rejected':int((~merged.accepted).sum()),
                       'MAE':float(np.mean(abs(e))) if len(e) else np.nan,'RMSE':float(np.sqrt(np.mean(e**2))) if len(e) else np.nan})
    pd.DataFrame(errors).to_csv(out/'manual_validation.csv',index=False);display(pd.DataFrame(errors))
else:
    stale=out/'manual_validation.csv'
    if stale.exists():stale.unlink()
    print('No manual measurements supplied; MAE/RMSE are pending.')
trial_files=list((cfg['_output_dir']/'module3').glob('*/physiological_parameters.csv'))
trials=pd.concat([pd.read_csv(p) for p in trial_files],ignore_index=True)
metrics=['latency_s','response_amplitude_px']
if 'response_amplitude_mm' in trials: metrics.append('response_amplitude_mm')
summary=trials.groupby('record_type')[metrics].agg(['count','mean','std'])
summary.to_csv(cfg['_output_dir']/'module3'/'trial_variability.csv')
display(summary)
print('Pixel amplitudes are comparable across trials only at the same image scale. Use calibrated mm values for different camera distances.')


## Discussion
Compare RGB texture descriptors on the approximate iris sample and justify the channel selection after inspecting artifacts. Discuss raw versus cleaned curves, rejected samples and effects of threshold/smoothing choices. Confirm segmentation with manual references before interpreting latency, amplitude or peak speed. Full-video ROI and seed automation for unseen recordings remain future work.